In [13]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Data,AmlCompute
from azure.ai.ml.constants import AssetTypes


In [8]:
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

Found the config file in: ./config.json


In [22]:
from azure.ai.ml import MLClient
from azure.identity import InteractiveBrowserCredential

# This will trigger a browser pop-up to log into your Azure account
credential = InteractiveBrowserCredential()

# Initialize MLClient
ml_client = MLClient.from_config(credential=credential)

Found the config file in: ./config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [10]:
compute_name = "batch-cluster"

In [15]:
try:
    compute_target = ml.client.compute.get(compute_name)
    print(f"Found existing compute target: {compute_name}")
except Exception as e:
    compute_target = AmlCompute(name=compute_name, 
                                size="STANDARD_D2_V2", 
                                type="AmlCompute",
                                min_instances=0,
                                max_instances=4,
                                idle_time_before_scale_down=120
                                )


In [24]:
from azure.ai.ml import MLClient
from azure.identity import InteractiveBrowserCredential

# Explicitly target your subscription's correct tenant ID
tenant_id = "32631e6c-2bce-4d34-8fbf-dcd501a37c39"

credential = InteractiveBrowserCredential(tenant_id=tenant_id)

# Initialize MLClient
ml_client = MLClient.from_config(credential=credential)
print("Successfully authenticated and connected to workspace!")

Found the config file in: ./config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Successfully authenticated and connected to workspace!


In [25]:
ml_client.compute.begin_create_or_update(compute_target).result()

AmlCompute({'type': 'amlcompute', 'created_on': '2026-10-14T07:46:32.4833115+00:00', 'provisioning_state': <ProvisioningState.SUCCEEDED: 'Succeeded'>, 'provisioning_errors': None, 'name': 'batch-cluster', 'description': None, 'tags': None, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/7ec95194-0f00-495d-aaa5-b1be9a0cd09e/resourceGroups/Leadscoringdeployement/providers/Microsoft.MachineLearningServices/workspaces/LeadScoring/computes/batch-cluster', 'Resource__source_path': '', 'base_path': '/Users/ram/Desktop/Machine Learning Models/Leadscoring', 'creation_context': None, 'resource_id': None, 'location': 'eastus2', 'size': 'STANDARD_D2_V2', 'min_instances': 0, 'max_instances': 4, 'idle_time_before_scale_down': 120.0, 'identity': None, 'ssh_public_access_enabled': True, 'ssh_settings': None, 'network_settings': <azure.ai.ml.entities._compute.compute.NetworkSettings object at 0x11a4c4bb0>, 'tier': 'dedicated', 'enable_node_public_ip': True, 'subnet': None})

In [32]:
## creating inference.py
import os
import joblib
import pandas as pd
def init():
    """Loads the model into memory once when the batch worker starts."""
    global model
    # AZUREML_MODEL_DIR points to your registered model directory
    model_dir = os.getenv("AZUREML_MODEL_DIR")
    model_path = os.path.join(model_dir, "model.joblib")
    model = joblib.load(model_path)

def run(mini_batch):
    """Processes incoming mini-batches of data files in parallel."""
    results = []
    for file_path in mini_batch:
        # Read each input data file (e.g., CSV)
        data = pd.read_csv(file_path)
        
        # Make predictions using your trained model
        predictions = model.predict(data)
        
        # Append predictions back to data frame
        df_out = data.copy()
        df_out["prediction"] = predictions
        results.append(df_out)
        
    return pd.concat(results, ignore_index=True)


In [33]:
from azure.ai.ml.entities import BatchEndpoint, BatchDeployment

In [34]:
endpoint_name = "lead-scoring-batch-endpoint"
deployment_name = "v1"

In [30]:
endpoint = BatchEndpoint(
    name=endpoint_name,
    description="Batch scoring endpoint for lead scoring",
)
ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

BatchEndpoint({'scoring_uri': 'https://lead-scoring-batch-endpoint.eastus2.inference.ml.azure.com/jobs', 'openapi_uri': None, 'provisioning_state': <EndpointProvisioningState.SUCCEEDED: 'Succeeded'>, 'name': 'lead-scoring-batch-endpoint', 'description': 'Batch scoring endpoint for lead scoring', 'tags': {}, 'properties': {'BatchEndpointCreationApiVersion': '2023-10-01', 'azureml.onlineendpointid': '/subscriptions/7ec95194-0f00-495d-aaa5-b1be9a0cd09e/resourceGroups/Leadscoringdeployement/providers/Microsoft.MachineLearningServices/workspaces/LeadScoring/batchEndpoints/lead-scoring-batch-endpoint'}, 'print_as_yaml': False, 'id': '/subscriptions/7ec95194-0f00-495d-aaa5-b1be9a0cd09e/resourceGroups/Leadscoringdeployement/providers/Microsoft.MachineLearningServices/workspaces/LeadScoring/batchEndpoints/lead-scoring-batch-endpoint', 'Resource__source_path': '', 'base_path': '/Users/ram/Desktop/Machine Learning Models/Leadscoring', 'creation_context': None, 'auth_mode': 'aad_token', 'location'

In [41]:
from azure.ai.ml.entities import ModelBatchDeployment

endpoint_name = "lead-scoring-batch-endpoint"
deployment_name = "v1"

# Use ModelBatchDeployment instead of BatchDeployment
deployment = ModelBatchDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    model="lead-scoring-model:1",  # Your registered model and version
    scoring_script="inference_azure.py",
    code_path="./",              # Path to the folder containing inference.py
    environment="AzureML-sklearn-1.5-ubuntu22.04-py310:1",
    compute="batch-cluster",      # Your compute cluster name
    instance_count=2,
    mini_batch_size=2,
    output_action="append_row",
)

# Submit the deployment to Azure
ml_client.batch_deployments.begin_create_or_update(deployment).result()
print(f"Batch endpoint '{endpoint_name}' and deployment '{deployment_name}' are ready!")

Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy '/Users/ram/Desktop/Machine Learning Models/Leadscoring' 'https://leadscoring2548308906.blob.core.windows.net/133af24a-810e-484d-aebe-60dec5eb4cd2-o759dfv8l9b275jhtn81yrwhik/Leadscoring' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Uploading Leadscoring (1460.38 MBs):  40%|███▉      | 583430505/1460375904 [10:31<15:49, 923980.34it/s]   


KeyboardInterrupt: 

In [4]:
import os

# Paste your actual connection string from the Azure Portal here
os.environ[
    "AZURE_STORAGE_CONNECTION_STRING"
] = "DefaultEndpointsProtocol=https;AccountName=leadscoring2548308906;AccountKey=Au2LFku8dwCUXdQ9AsHsS6agRxooYDdxK4M5Fx1Hs34b8CDKbC9Wb7VYSVm8/J0w6BJjQHvAseKQ+AStPcqDAw==;EndpointSuffix=core.windows.net"

In [5]:
import os 
import pandas as pd
import joblib
import pyodbc
from azure.storage.blob import BlobServiceClient




# # Load model from azure blob storage
def load_model():
    connect_str = os.getenv('AZURE_STORAGE_CONNECTION_STRING')
    blob_service_client = BlobServiceClient.from_connection_string(connect_str)
    blob_client = blob_service_client.get_blob_client(container='ml-model-container-leadscoring', blob='models/lead_scoring/best_xg_model.joblib')
    model_path = "/tmp/model.joblib"
    with open(model_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())
    
    return joblib.load(model_path)

In [6]:
LOAD_MODEL = load_model()

In [19]:
import pandas as pd


def load_test_data():
  # Define the file paths on your Mac
  x_test_path = (
      "/Users/ram/Desktop/Machine Learning Models/Leadscoring/X_test.csv"
  )
  y_test_path = (
      "/Users/ram/Desktop/Machine Learning Models/Leadscoring/Y_test.csv"
  )

  # Read the CSV files into pandas DataFrames
  X_test = pd.read_csv(x_test_path)
  Y_test = pd.read_csv(y_test_path)

  return X_test, Y_test


# --- How to use it in your script or notebook ---
if __name__ == "__main__":
  # 1. Load your test data locally
  X_test, Y_test = load_test_data()

  # 2. Load your model (either from local or blob storage)
  # model = joblib.load("model.joblib")

  # 3. Make predictions
  # predictions = model.predict(X_test)
  # print(f"Successfully generated predictions for {len(X_test)} rows.")

In [21]:
Y_test

,Converted
0,1
1,0
2,0
3,0
4,0
...,...
2583,1
2584,0
2585,1
2586,0


In [23]:
def update_lead_scores(df_scored):
    conn_str = os.getenv("AZURE_SQL_CONNECTION_STRING")
    with pyodbc.connect(conn_str) as conn:
        cursor = conn.cursor()
        for _, row in df_scored.iterrows():
            cursor.execute(
                "UPDATE Leads SET lead_score = ?, scored_at = GETDATE() WHERE lead_id = ?",
                float(row['predicted_score']),
                row['lead_id']
            )
        conn.commit()

In [ ]:
if __name__ == "__main__":
    model = load_model()
    leads_df = X_test
    
    if not leads_df.empty:
        features = leads_df[['feature_1', 'feature_2', 'feature_3']]
        # Assuming classification probability or regression score
        leads_df['predicted_score'] = model.predict_proba(features)[:, 1] 
        update_lead_scores(leads_df)
        print(f"Successfully scored {len(leads_df)} leads.")
    else:
        print("No new leads to score.")

In [8]:
X_TEST = pd.read_csv("/Users/ram/Desktop/Machine Learning Models/Leadscoring/X_test.csv")

In [10]:
Y_test = pd.read_csv("/Users/ram/Desktop/Machine Learning Models/Leadscoring/Y_test.csv")

In [9]:
PRED = LOAD_MODEL.predict(X_TEST)

In [12]:
accuracy = (PRED == Y_test.values.flatten()).mean()
accuracy

np.float64(0.866692426584235)